In [1]:
# We use sqlite3 to interact with the database and pandas to handle our CSV data
import sqlite3
import pandas as pd

Create an in-memory SQLite database instance locally.

In [2]:
# Connect to an in-memory SQLite database
con = sqlite3.connect(':memory:')

# Create a cursor object to execute SQL commands
cur = con.cursor()

In [3]:
# Define a helper function to execute an entire SQL script file
def execute_sql_script_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            sql_script = f.read()

        print(f'🖥 Executing SQL script from file {filepath}')
        cur.executescript(sql_script)
        print(f"✅ Successfully executed SQL script from file {filepath}")
    except Exception as e:
        print(f"❌ An error occurred: {e}")


def execute_sql_query(query):
    try:
        # Execute the query
        cur.execute(query)

        if cur.description is not None:
            # Fetch all rows and column names
            rows = cur.fetchall()
            column_names = [description[0] for description in cur.description]
    
            if rows:
                print(f"✅ Query executed successfully. {len(rows)} row{'s' if len(rows) >= 2 else ''} to display.")
            else:
                print("✅ Query executed successfully. No results to display.")
    
            df = pd.DataFrame(rows, columns=column_names)
            display(df.style.hide(axis='index'))
        else:
            # For INSERT, UPDATE, DELETE, etc.
            print("✅ Query executed successfully. No results to display.")
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")

In [4]:
execute_sql_script_from_file('./sql-scripts/01_create_tables.sql')

🖥 Executing SQL script from file ./sql-scripts/01_create_tables.sql
✅ Successfully executed SQL script from file ./sql-scripts/01_create_tables.sql


In [5]:
execute_sql_script_from_file('./sql-scripts/02_populate_tables.sql')

🖥 Executing SQL script from file ./sql-scripts/02_populate_tables.sql
✅ Successfully executed SQL script from file ./sql-scripts/02_populate_tables.sql


In [6]:
execute_sql_query('SELECT * FROM departments;')

✅ Query executed successfully. 3 rows to display.


DepartmentID,DepartmentName
1,Engineering
2,Human Resources
3,Marketing


In [7]:
execute_sql_query('SELECT * FROM job_titles;')

✅ Query executed successfully. 4 rows to display.


JobTitleID,JobTitle
1,Software Engineer
2,HR Specialist
3,Marketing Manager
4,Data Analyst


In [8]:
execute_sql_query('SELECT * FROM employees;')

✅ Query executed successfully. 5 rows to display.


EmployeeID,FullName,StartDate,EndDate,DepartmentID,JobTitleID,Status,PayType,PayRate
101,Alice Kim,2022-01-10,None,1,1,Active,Salary,85000.000000
102,Bob Lee,2023-03-15,None,1,4,Active,Salary,70000.000000
103,Cindy Park,2024-07-01,None,2,2,On-boarding,Salary,60000.000000
104,David Tran,2021-05-21,None,3,3,Active,Salary,95000.000000
105,Emily Zhou,2020-11-18,2023-12-15,3,4,Terminated,Hourly,45.000000


## Using SQL to impose data integrity

### Prevent duplicate `DepartmentID`

In [9]:
execute_sql_query('''
    INSERT INTO departments (DepartmentID, DepartmentName) 
    VALUES (1, 'Sales'); -- ⚠️ DepartmentID == 1 already exists in the departments table
''')

An error occurred: UNIQUE constraint failed: departments.DepartmentID


#### Correct Query

Note: Running this query multiple times will keep adding an extra row to the `departments` table with duplicate `DepartmentName` value (`'Sales'`). This is because we have not added a `UNIQUE` constraint to the `DepartmentName` column.

In [15]:
execute_sql_query('''
    INSERT INTO departments (DepartmentName) 
    VALUES ('Sales');
''')

An error occurred: UNIQUE constraint failed: departments.DepartmentID


In [11]:
execute_sql_query('''
    SELECT * FROM departments;
''')

✅ Query executed successfully. 4 rows to display.


DepartmentID,DepartmentName
1,Engineering
2,Human Resources
3,Marketing
4,Sales


### Prevent duplicate `JobTitle`

In [16]:
execute_sql_query('''
    INSERT INTO job_titles (JobTitle) 
    VALUES ('Software Engineer'); -- ⚠️ `Software Engineer` already exists in the 
''')

An error occurred: UNIQUE constraint failed: job_titles.JobTitle


### Prevent negative `PayRate`

In [12]:
execute_sql_query('''
    INSERT INTO employees (
        EmployeeID,
        FullName,
        StartDate,
        EndDate,
        DepartmentID,
        JobTitleID,
        Status,
        PayType,
        PayRate
    ) VALUES (
        108,
        'Ivan Choi',
        '2023-01-01',
        NULL,
        1,
        1,
        'Active',
        'Hourly',
        -25.00 -- ⚠️ PayRate is a negative amount
    );
''')

An error occurred: CHECK constraint failed: PayRate > 0


🔍 Why it fails: `PayRate` < 0 violates the `CHECK` constraint (`PayRate` > 0)

### Prevent `EndDate` earlier than `StartDate`

In [13]:
execute_sql_query('''
    INSERT INTO employees (
        EmployeeID,
        FullName,
        StartDate,
        EndDate,
        DepartmentID,
        JobTitleID,
        Status,
        PayType,
        PayRate
    ) VALUES (
        109,
        'Jenny Park',
        '2024-01-01',
        '2022-12-01', -- ⚠️ EndDate is 2022, while the start date is 2024
        1,
        1,
        'Terminated',
        'Salary',
        80000
    );
''')

An error occurred: CHECK constraint failed: EndDate IS NULL OR EndDate >= StartDate


🔍 Why it fails: `EndDate` < `StartDate` → `CHECK` constraint violation (`EndDate IS NULL OR EndDate >= StartDate`)

### Prevent invalid employee `Status`

In [14]:
execute_sql_query('''
    INSERT INTO employees (
        EmployeeID,
        FullName,
        StartDate,
        EndDate,
        DepartmentID,
        JobTitleID,
        Status,
        PayType,
        PayRate
    ) VALUES (
        107,
        'Hannah Yoo',
        '2023-09-01',
        NULL,
        1,
        1,
        'Actiive', -- ⚠️ Invalid status (typo) - should be 'Active' instead of 'Actiive'
        'Salary',
        75000
    );
''')

An error occurred: CHECK constraint failed: Status IN ('Active', 'On-boarding', 'Terminated')


🔍 Why it fails: 'Actiive' is not one of the allowed values → `CHECK` constraint on `Status` (`IN ('Active', 'On-boarding', 'Terminated')`)